In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score,log_loss

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

**Can calendar signals help predict next-day stock direction for Apple?**

In [40]:
df = pd.read_csv("Apple Data.csv")

In [41]:
df = df.replace(",", "", regex=True)
#Converting non-date columns to float->for use in ML models
for col in df.columns:
    if col != "Date":
        df[col] = df[col].astype(float)


In [42]:
df

,Date,Open,High,Low,Close,Adj_Close,Volume
0,23-Jan-26,247.32,249.41,244.68,248.04,248.04,41625700.0
1,22-Jan-26,249.20,251.00,248.15,248.35,248.35,39708300.0
2,21-Jan-26,248.70,251.56,245.18,247.65,247.65,54641700.0
3,20-Jan-26,252.73,254.79,243.42,246.70,246.70,80267500.0
4,16-Jan-26,257.90,258.90,254.93,255.53,255.53,72142800.0
...,...,...,...,...,...,...,...
1251,29-Jan-21,135.83,136.74,130.21,131.96,128.46,177523800.0
1252,28-Jan-21,139.52,141.99,136.70,137.09,133.45,142621100.0
1253,27-Jan-21,143.43,144.30,140.41,142.06,138.29,140843800.0
1254,26-Jan-21,143.60,144.30,141.37,143.16,139.36,98390600.0


In [43]:
df.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume'], dtype='object')

In [44]:
df = df.dropna().reset_index(drop=True)

In [45]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

/tmp/ipython-input-171143142.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])


**Feature Engineering**

In [46]:
df["return"] = df["Adj_Close"].pct_change()
df["range_pct"] = (df["High"] - df["Low"]) / df["Open"]
df["gap"] = df["Open"] / df["Adj_Close"].shift(1) - 1

df["ret_5"] = df["return"].rolling(5).mean()
df["ret_10"] = df["return"].rolling(10).mean()

df["vol_5"] = df["return"].rolling(5).std()
df["vol_10"] = df["return"].rolling(10).std()

df["vol_change"] = df["Volume"].pct_change()
df["weekday"] = df["Date"].dt.dayofweek

df["return_1d"] = df["Close"].pct_change()
df["range"] = (df["High"] - df["Low"]) / df["Close"]
df["volume_z"] = ((df["Volume"] - df["Volume"].rolling(20).mean()) /
    df["Volume"].rolling(20).std()
)


In [47]:
df["target"] = (df["Adj_Close"].shift(-1) > df["Adj_Close"]).astype(int)
#1 → next day is up
#0 → next day is down or flat

In [48]:
features = [
    # Returns  or momentum
    "return",
    "ret_5",
    "ret_10",

    # Volatility - 5 and 10 days
    "vol_5",
    "vol_10",
    # Price structure
    "range_pct",
    "gap",
    "range",
    # Volume
    "vol_change",
    "volume_z",
    # Calendar
    "weekday",
    # Short-term context
    "return_1d"
]

X = df[features]
y = df["target"]

**Model Building**

In [49]:
tscv = TimeSeriesSplit(n_splits=5)

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=6,
        min_samples_leaf=40,
        random_state=42
    ))
])


gb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model
}
#imputer to deal with NAs/NULL

**Evaluation**

In [50]:
def evaluate_model(model, X, y, cv):
    accs, aucs, losses = [], [], []

    for train_idx, test_idx in cv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        aucs.append(roc_auc_score(y_test, probs))
        losses.append(log_loss(y_test, probs))

    return np.mean(accs), np.mean(aucs), np.mean(losses)


**Accuracy** → Fraction of days the model gets the direction right (intuitive, but noisy).

**ROC-AUC**→ Measures whether the model ranks up-days above down-days (best indicator of weak financial signal). Measures ability to distinguish between classes.

**Log Loss** → Evaluates probability quality and penalizes overconfident wrong predictions (critical for trading).

In [51]:
print("Model Performance (Time-Series CV):\n")

for name, model in models.items():
    acc, auc, ll = evaluate_model(model, X, y, tscv)
    print(
        f"{name:20s} | "
        f"Accuracy: {acc:.4f} | "
        f"AUC: {auc:.4f} | "
        f"LogLoss: {ll:.4f}"
    )


Model Performance (Time-Series CV):

Logistic Regression  | Accuracy: 0.5033 | AUC: 0.4803 | LogLoss: 0.7035
Random Forest        | Accuracy: 0.4967 | AUC: 0.4970 | LogLoss: 0.6950
Gradient Boosting    | Accuracy: 0.4919 | AUC: 0.4939 | LogLoss: 0.8326
